# Statistical Comparison of Multilayer Networks

This notebook demonstrates how to use py3plex's statistical comparison framework to compare multilayer networks.

## Overview

The `compare_multilayer_networks` function enables:
- Pairwise and multi-group comparisons
- Multiple statistical tests (parametric, non-parametric, permutation)
- Effect size estimation
- Multiple comparison correction
- Bootstrap confidence intervals

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from py3plex.core import multinet
from py3plex.algorithms.statistics.stats_comparison import (
    compare_multilayer_networks,
    bootstrap_confidence_interval
)

## Example 1: Basic Two-Network Comparison

Let's create two simple multilayer networks and compare them.

In [ ]:
# Create Network 1: Dense social network
net1 = multinet.multi_layer_network(directed=False)
net1.add_edges([
    # Facebook layer - dense connections
    ['Alice', 'facebook', 'Bob', 'facebook', 1],
    ['Alice', 'facebook', 'Carol', 'facebook', 1],
    ['Bob', 'facebook', 'Carol', 'facebook', 1],
    ['Bob', 'facebook', 'David', 'facebook', 1],
    ['Carol', 'facebook', 'David', 'facebook', 1],
    # Twitter layer
    ['Alice', 'twitter', 'Bob', 'twitter', 1],
    ['Alice', 'twitter', 'Carol', 'twitter', 1],
    ['Bob', 'twitter', 'David', 'twitter', 1],
    # Inter-layer connections
    ['Alice', 'facebook', 'Alice', 'twitter', 1],
    ['Bob', 'facebook', 'Bob', 'twitter', 1],
], input_type='list')

print(f"Network 1: {len(list(net1.get_nodes()))} nodes, {len(list(net1.get_edges()))} edges")

In [ ]:
# Create Network 2: Sparse social network
net2 = multinet.multi_layer_network(directed=False)
net2.add_edges([
    # Facebook layer - sparse
    ['Alice', 'facebook', 'Bob', 'facebook', 1],
    ['Carol', 'facebook', 'David', 'facebook', 1],
    # Twitter layer
    ['Alice', 'twitter', 'Carol', 'twitter', 1],
    ['Bob', 'twitter', 'David', 'twitter', 1],
    # Inter-layer connections
    ['Alice', 'facebook', 'Alice', 'twitter', 1],
], input_type='list')

print(f"Network 2: {len(list(net2.get_nodes()))} nodes, {len(list(net2.get_edges()))} edges")

### Compare Networks Using Permutation Test

In [ ]:
# Perform comparison
results = compare_multilayer_networks(
    [net1, net2],
    metrics=['density', 'average_degree', 'node_activity'],
    test='permutation',
    n_permutations=1000,
    correction='fdr_bh',
    alpha=0.05
)

print("\nComparison Results:")
print(results.to_string())

In [ ]:
# Show only significant differences
significant = results[results['significant']]
if len(significant) > 0:
    print("\nStatistically Significant Differences:")
    print(significant[['metric', 'layer', 'p_value', 'adjusted_p_value', 'effect_size']].to_string())
else:
    print("\nNo statistically significant differences found.")

## Example 2: Multiple Network Comparison with ANOVA

Compare three or more networks using ANOVA or Kruskal-Wallis test.

In [ ]:
# Create Network 3
net3 = multinet.multi_layer_network(directed=False)
net3.add_edges([
    ['Alice', 'facebook', 'Bob', 'facebook', 1],
    ['Bob', 'facebook', 'Carol', 'facebook', 1],
    ['Carol', 'facebook', 'David', 'facebook', 1],
    ['Alice', 'twitter', 'Bob', 'twitter', 1],
    ['Bob', 'twitter', 'Carol', 'twitter', 1],
    ['Alice', 'facebook', 'Alice', 'twitter', 1],
    ['Bob', 'facebook', 'Bob', 'twitter', 1],
    ['Carol', 'facebook', 'Carol', 'twitter', 1],
], input_type='list')

# Compare three networks
results_multi = compare_multilayer_networks(
    [net1, net2, net3],
    metrics=['density', 'average_degree'],
    test='kruskal',  # Non-parametric for 3+ groups
    correction='holm',
    alpha=0.05
)

print("\nThree-Network Comparison:")
print(results_multi.to_string())

## Example 3: Directed Networks

The framework also works with directed multilayer networks.

In [ ]:
# Create directed network 1
dir_net1 = multinet.multi_layer_network(directed=True)
dir_net1.add_edges([
    ['A', 'L1', 'B', 'L1', 1],
    ['B', 'L1', 'C', 'L1', 1],
    ['C', 'L1', 'A', 'L1', 1],  # Cycle
    ['A', 'L2', 'B', 'L2', 1],
], input_type='list')

# Create directed network 2
dir_net2 = multinet.multi_layer_network(directed=True)
dir_net2.add_edges([
    ['A', 'L1', 'B', 'L1', 1],
    ['B', 'L1', 'C', 'L1', 1],  # No cycle
    ['A', 'L2', 'B', 'L2', 1],
], input_type='list')

# Compare directed networks
results_directed = compare_multilayer_networks(
    [dir_net1, dir_net2],
    metrics=['density', 'average_degree'],
    test='mann-whitney',
    alpha=0.05
)

print("\nDirected Network Comparison:")
print(results_directed.to_string())

## Example 4: Bootstrap Confidence Intervals

Estimate confidence intervals for metrics using bootstrap resampling.

In [ ]:
# Define a custom metric function
def average_density(network):
    """Calculate average density across all layers."""
    from py3plex.algorithms.statistics import multilayer_statistics as mls
    from py3plex.algorithms.statistics.stats_comparison import _get_layers
    
    layers = _get_layers(network)
    if not layers:
        return 0.0
    densities = [mls.layer_density(network, layer) for layer in layers]
    return np.mean(densities)

# Compute bootstrap confidence intervals
ci = bootstrap_confidence_interval(
    [net1, net2],
    average_density,
    n_bootstrap=1000,
    confidence_level=0.95
)

print("\n95% Bootstrap Confidence Intervals for Average Density:")
for group, (lower, upper) in ci.items():
    print(f"{group}: [{lower:.4f}, {upper:.4f}]")

## Example 5: Different Statistical Tests

Demonstrate various statistical tests available.

In [ ]:
# Compare using different tests
test_types = ['permutation', 't-test', 'mann-whitney']

for test_type in test_types:
    results_test = compare_multilayer_networks(
        [net1, net2],
        metrics=['density'],
        test=test_type,
        n_permutations=500 if test_type == 'permutation' else 1000,
        alpha=0.05
    )
    
    print(f"\n{test_type.upper()} Results:")
    print(results_test[['metric', 'layer', 'statistic', 'p_value', 'effect_size']].to_string())

## Example 6: Multiple Comparison Correction

When testing multiple metrics, apply correction for multiple comparisons.

In [ ]:
# Test many metrics
all_metrics = ['density', 'average_degree', 'clustering', 'node_activity', 'entropy']

correction_methods = ['bonferroni', 'holm', 'fdr_bh']

for correction in correction_methods:
    results_corr = compare_multilayer_networks(
        [net1, net2],
        metrics=all_metrics,
        test='permutation',
        n_permutations=500,
        correction=correction,
        alpha=0.05
    )
    
    n_significant = results_corr['significant'].sum()
    print(f"\n{correction.upper()}: {n_significant} significant results out of {len(results_corr)}")
    
    if n_significant > 0:
        print(results_corr[results_corr['significant']][['metric', 'p_value', 'adjusted_p_value']].to_string())

## Example 7: Synthetic Network Benchmark

Create synthetic networks with known differences to validate the framework.

In [ ]:
def create_random_multilayer_network(n_nodes, n_layers, edge_prob, directed=False):
    """Create a random multilayer network."""
    net = multinet.multi_layer_network(directed=directed)
    nodes = [f'N{i}' for i in range(n_nodes)]
    layers = [f'L{i}' for i in range(n_layers)]
    
    # Add intra-layer edges
    edges = []
    for layer in layers:
        for i in range(n_nodes):
            for j in range(i+1, n_nodes):
                if np.random.random() < edge_prob:
                    edges.append([nodes[i], layer, nodes[j], layer, 1])
    
    # Add inter-layer edges
    for i in range(len(layers)-1):
        for node in nodes:
            if np.random.random() < 0.5:
                edges.append([node, layers[i], node, layers[i+1], 1])
    
    if edges:
        net.add_edges(edges, input_type='list')
    
    return net

# Create two groups of networks with different densities
np.random.seed(42)
group1_nets = [create_random_multilayer_network(10, 2, 0.3) for _ in range(3)]
group2_nets = [create_random_multilayer_network(10, 2, 0.1) for _ in range(3)]

# Compare the groups
results_synthetic = compare_multilayer_networks(
    group1_nets + group2_nets,
    metrics=['density', 'average_degree'],
    test='anova',
    alpha=0.05
)

print("\nSynthetic Network Comparison:")
print("Group 1: Higher density (p=0.3)")
print("Group 2: Lower density (p=0.1)")
print("\nResults:")
print(results_synthetic.to_string())

## Summary

This notebook demonstrated:

1. **Basic two-network comparison** using permutation tests
2. **Multi-network comparison** with ANOVA/Kruskal-Wallis
3. **Directed network** comparisons
4. **Bootstrap confidence intervals** for metric uncertainty
5. **Different statistical tests** and their appropriate use cases
6. **Multiple comparison correction** methods
7. **Synthetic benchmarks** for validation

### Key Recommendations:

- For small samples (n < 30): Use **permutation** or **non-parametric** tests
- For normal distributions: Use **parametric** tests (more power)
- For multiple metrics: Apply **correction** (FDR recommended)
- Always report **effect sizes** in addition to p-values
- Use **bootstrap CI** to quantify uncertainty

### Available Metrics:

- `density`: Layer density
- `average_degree`: Mean node degree
- `clustering`: Clustering coefficient
- `node_activity`: Fraction of layers where nodes are active
- `coupling_strength`: Inter-layer connection strength
- `entropy`: Entropy of multiplexity

### Available Tests:

- `permutation`: Non-parametric permutation test (2+ groups)
- `t-test`: Student's t-test (2 groups, parametric)
- `mann-whitney`: Mann-Whitney U (2 groups, non-parametric)
- `anova`: One-way ANOVA (2+ groups, parametric)
- `kruskal`: Kruskal-Wallis H (2+ groups, non-parametric)